# Workbook 3 — Why LLMs?

**Solutions · Mysteries of Deep Learning, Iliad Intensive 2026**

Deep learning reached large language models through a series of choices, each of which looked contingent at the time: learn without labels, learn from text, predict the next token, use the transformer, and make it big. This workbook takes those choices one at a time. The last section asks what the resulting object *is*, and what reinforcement learning on top of it does.

Where there is something to derive, a section gives the setup, an exercise with hints, and a worked answer. Where a part is about a reading, it gives a sentence or two on what the reading shows and the link, labelled *core* or *optional*, so you can decide whether to read it. In the exercises variant the answer cells are blank; the solutions variant has worked answers, and both are available to everyone. Sections are independent, so start wherever your interest is. Nobody is expected to do all of it.

**Route**
1. [Why generative, why unsupervised](#wb3-section-1). LeCun's cake, and a comparison of code lengths for labels and raw text. One counting exercise.
2. [Why text? Why not video?](#wb3-section-2). Reading only; the shortest section. LeCun's case for predicting in a learned representation space instead of predicting raw observations.
3. [Why next-token prediction?](#wb3-section-3). Cross-entropy is code length. One exercise derives the arithmetic-coding bound; an optional one asks why a text model can compress images. A pointer to the prequential code and Solomonoff (WB4).
4. [A hundred flowers and the coming of the transformer](#wb3-section-4). Symmetry and locality in CNNs and the geometric-deep-learning blueprint (symmetry plus scale separation), with two short symmetry exercises; readings on the transformer and the hardware lottery.
5. [Why scale? Should *you* consume more tokens?](#wb3-section-5). Kaplan and Chinchilla. Three exercises derive the compute-optimal allocation and plug in Chinchilla's constants (5.1), explain why Kaplan's answer differed (5.2), and compare training with inference cost (5.3).
6. [Simulators, personae, and RL on top](#wb3-section-6). Readings on base models as simulators and on the Waluigi effect; two exercises on the closed-form optimum of KL-regularized RLHF and what it does to a mixture of personae; readings on how far RL moves a model. This section stands alone if alignment is what you came for.
7. [What carries forward](#wb3-closing). Pointers to the other workbooks.

<a id="wb3-section-1"></a>

## 1. Why generative, why unsupervised

**The result.** A label for a $k$-way classifier can be encoded in at most $\log_2k$ bits, while raw data take as many bits to encode as they still surprise a predictive model by. Exercise 1.1 compares the totals for ImageNet's labels and a trillion tokens of text. This compares code lengths; it does not measure how much useful knowledge a learner extracts.

**[LeCun, "Predictive learning" (NIPS 2016 invited talk)](https://nips.cc/virtual/2016/invited-talk/6197)**, Yann LeCun, 2016 · *core, short*

The cake talk. Argues that most of what an intelligent system learns must come from predicting its own input, because that is where most of the bits are; supervision and reward are the icing and the cherry. The talk is also on [YouTube](https://www.youtube.com/watch?v=Ount2Y4qxQo). If you would rather read, Sebastian Ruder's conference writeup ([Highlights of NIPS 2016](https://www.ruder.io/highlights-nips-2016/)) describes the cake and the reaction to it in a paragraph.

### 1.A. Exercise: code lengths of labels and of text

**Exercise 1.1.** Compare how many bits it takes to encode the targets a learner is asked to predict.

(a) ImageNet has about $1.28\times10^6$ training images in $k=1000$ classes. Bound the number of bits needed to encode all its labels, in bits and in megabytes.

(b) A language model is trained on $T=10^{12}$ tokens with a vocabulary of $V\approx5\times10^4$. Give the crude upper bound $T\log_2V$. Then give a better estimate: the number of bits by which the text still *surprises* a trained model whose cross-entropy is about $2.5$ nats per token (an illustrative value).

(c) What is the ratio of your estimate in (b) to (a)?

(d) How many raw bits are in one $224\times224$ RGB image at 8 bits per channel? LeCun's slide credits RL with "a few bits for some samples", supervision with "10→10,000 bits per sample" and prediction with "millions of bits per sample". Which of these does your count match?

*Hint for (b).* Cross-entropy in nats is the average of $-\ln q(x_t\mid x_{<t})$ over tokens; divide by $\ln2$ to convert to bits.

**(a)** An arbitrary label from 1000 classes can be encoded in at most $\log_21000\approx9.97$ idealized bits. Multiplying gives an upper bound on the entropy of the label sequence:
$$
1.28\times10^6\times9.97\approx1.3\times10^7\ \text{bits}\approx1.6\ \text{MB}.
$$

**(b)** Crude bound: $\log_2(5\times10^4)\approx15.6$ bits per token, so $10^{12}\times15.6\approx1.6\times10^{13}$ bits. Residual surprise: $2.5/\ln2\approx3.6$ bits per token, so
$$
10^{12}\times\frac{2.5}{\ln2}\approx3.6\times10^{12}\ \text{bits}.
$$
This is a model-dependent code-length estimate, including irreducible surprise; it does not measure how much useful information training puts into the parameters.

**(c)** $3.6\times10^{12}/1.3\times10^{7}\approx3\times10^{5}$, between five and six orders of magnitude. Since (a) is an upper bound, this ratio is a lower bound for these assumptions.

**(d)** $224\times224\times3\times8\approx1.2\times10^{6}$ bits: the prediction layer's "millions of bits per sample". As in (b), this is raw code length, and most of these bits are irrelevant to any particular downstream task. §2 asks why, given this, text rather than video.

<a id="wb3-section-2"></a>

## 2. Why text? Why not video?

Unsettled. For text: nearly every bit was put there by a mind, so predicting it may force a model to model minds (janus's "Simulators", §6, develops this idea). For video: far more bits, grounded in physics.

**[A path towards autonomous machine intelligence](https://openreview.net/forum?id=BZ5a1r-kVsf)**, Yann LeCun, 2022 (position paper, v0.9.2) · *optional*

Argues that generative, likelihood-based prediction of raw observations is the wrong objective for learning world models; instead, learn to predict in a learned representation space (JEPA), trained with non-contrastive, energy-based objectives, as the core of a modular agent with a world model, cost and planner. A long position paper (about 60 pages), not an empirical result. The introduction and the section introducing JEPA give the argument.

<a id="wb3-section-3"></a>

## 3. Why next-token prediction?

**The result.** The cross-entropy loss of a model $q$ on a sequence is, to within two bits, the length of the code for that sequence that arithmetic coding builds from $q$. So training a language model *is* training a compressor, and the excess expected loss over the data's entropy is exactly $\mathrm{KL}(p\Vert q)$. Next-token factorization is not a restriction: every distribution over sequences factorizes this way.

### 3.A. The factorization is free

Any distribution over sequences factorizes by the chain rule:

$$
q(x_{1:T})=\prod_{t=1}^{T}q(x_t\mid x_{<t}).
$$

So predicting the next token is not a modelling assumption; it is a parametrization. What *is* a choice is the left-to-right order and the decision to share one network across all positions. The payoff is that the loss decomposes into a sum of per-token terms, every token is a training target, and sampling is easy.

### 3.B. Exercise: cross-entropy is code length

**[Language modeling is compression](https://arxiv.org/abs/2309.10668)**, Grégoire Delétang, Anian Ruoss, et al. (DeepMind), 2023 · *optional; §2 for Exercise 3.1*

Uses large language models as the predictor inside an arithmetic coder. Fed raw bytes, they compress text, and also image patches and audio, better than gzip, PNG and FLAC, until the model's own size is counted. §2 (Background) sets out the equivalence of prediction and compression and describes arithmetic coding, the construction Exercise 3.1 works through. Exercise 3.2 asks about the image result.

Arithmetic coding turns any next-token model $q$ into a lossless code. Fix the sequence length $T$. Start with the interval $[0,1)$. At step $t$, split the current interval into consecutive sub-intervals, one per vocabulary item, with widths proportional to $q(\cdot\mid x_{<t})$, and keep the sub-interval belonging to the actual token $x_t$. Call the final interval $J(x_{1:T})$. A binary string $c=c_1\dots c_\ell$ names the dyadic interval $I(c)=[0.c_1\dots c_\ell,\ 0.c_1\dots c_\ell+2^{-\ell})$ (binary fractions). The codeword for $x_{1:T}$ is a string $c$ with $I(c)\subseteq J(x_{1:T})$.

**Exercise 3.1.**

(a) Show that $J(x_{1:T})$ has width $w=q(x_{1:T})=\prod_tq(x_t\mid x_{<t})$, and that distinct sequences have disjoint intervals.

(b) Show that any interval $[a,a+w)\subseteq[0,1)$ contains a dyadic interval of width $2^{-\ell}$ with $\ell=\lceil-\log_2w\rceil+1$.

(c) Show that the resulting code is prefix-free and that its length satisfies
$$
\ell(x_{1:T})<-\log_2q(x_{1:T})+2=\sum_{t=1}^{T}-\log_2q(x_t\mid x_{<t})+2\ \text{bits}.
$$
How large is the overhead per token for a long sequence?

(d) Let the data come from a distribution $p$. Show that $\mathbb E_p[-\log q(X)]=H(p)+\mathrm{KL}(p\Vert q)$, and conclude that, up to the two bits, the expected code length is minimized by $q=p$.

*Hints.* (a) Induct on $t$; the sub-intervals of one interval are disjoint. (b) Take $m=\lceil a2^{\ell}\rceil$ and check that $[m2^{-\ell},(m+1)2^{-\ell})$ fits. (c) A string $c$ is a prefix of $c'$ exactly when $I(c')\subseteq I(c)$. (d) Write $\log\frac1q=\log\frac1p+\log\frac pq$ and use $\mathrm{KL}\ge0$.

**(a)** By induction on $t$. Suppose the interval after step $t-1$ has width $q(x_{<t})$. It is split in proportion to $q(\cdot\mid x_{<t})$, so the kept piece has width $q(x_{<t})\,q(x_t\mid x_{<t})=q(x_{\le t})$. After $T$ steps, $w=q(x_{1:T})$. Two distinct sequences agree up to some first position $t$ where they differ. At that step they land in different, disjoint sub-intervals of the same interval, and every later interval is nested inside those, so the final intervals are disjoint.

**(b)** Let $m=\lceil a2^\ell\rceil$. Then $m2^{-\ell}\ge a$ and $m2^{-\ell}<a+2^{-\ell}$, so $(m+1)2^{-\ell}<a+2\cdot2^{-\ell}$. Since $\ell\ge-\log_2w+1$, we have $2\cdot2^{-\ell}\le w$, hence $(m+1)2^{-\ell}<a+w$. So $[m2^{-\ell},(m+1)2^{-\ell})\subseteq[a,a+w)$. Because $a+w\le1$, $m<2^\ell$, and the $\ell$-bit binary expansion of $m$ is the codeword.

**(c)** *Prefix-free:* if the codeword $c$ for $x$ were a prefix of the codeword $c'$ for some $x'\ne x$, then $I(c')\subseteq I(c)\subseteq J(x)$ and also $I(c')\subseteq J(x')$, contradicting the disjointness in (a).

*Length:* $\ell=\lceil-\log_2w\rceil+1<-\log_2w+2$, and $-\log_2w=-\log_2q(x_{1:T})=\sum_t-\log_2q(x_t\mid x_{<t})$ by the chain rule.

*Decoding:* the decoder runs the same model, makes the same splits, and at each step emits the token whose sub-interval contains $I(c)$.

The overhead is two bits for the *whole sequence*, so $2/T$ bits per token, negligible for long sequences. The model's total cross-entropy on a sequence, in bits, is its code length to within two bits. Conversely, the Kraft inequality says any prefix code satisfies $\sum_x2^{-\ell(x)}\le1$, so $2^{-\ell(x)}$ is a (sub-)probability distribution. Codes and models are the same object, up to rounding.

**(d)**
$$
\mathbb E_p[-\log q(X)]=\sum_xp(x)\log\frac1{p(x)}+\sum_xp(x)\log\frac{p(x)}{q(x)}=H(p)+\mathrm{KL}(p\Vert q).
$$
$\mathrm{KL}(p\Vert q)\ge0$, with equality iff $q=p$ (Gibbs' inequality, from Jensen's inequality applied to $\log$). By (c), the expected code length of arithmetic coding with $q$, in bits, lies below $H(p)+\mathrm{KL}(p\Vert q)+2$ (with both terms measured in bits), and no prefix code can beat $H(p)$ on average. So the excess expected loss of a model over the entropy of the data is exactly a KL divergence. Held-out cross-entropy is the quantity the scaling laws of §5 fit.

### 3.C. Bridge to WB4

Training itself can be read as compression. If you train online in one pass, predicting each new batch before you update on it, then the sum of the losses is the code length of the whole dataset under an adaptive coder (the *prequential* code). A model that learns fast compresses the data well. It is loosely analogous to Solomonoff induction, which predicts with a mixture over all programs weighted by $2^{-\text{length}}$; WB4 §3 covers Solomonoff induction itself.

### 3.D. Optional exercise: a text model as an image compressor

**Exercise 3.2 (optional).** Delétang et al. (reading in §3.B) use Chinchilla 70B, trained mainly on text, as the model inside an arithmetic coder. Fed small greyscale ImageNet patches and LibriSpeech audio as raw bytes, it compresses them further than PNG and FLAC do. Each dataset is 1 GB; count the model at 2 bytes per parameter. Using Exercise 3.1, explain why this works at all. Then say what changes if the size of the model is counted as part of the compressed output, as a fair comparison of compressors requires.

- **Why it works.** Arithmetic coding works with *any* $q$: nothing in Exercise 3.1 used the data the model was trained on. A text model is a general sequence model, and within its context window it can pick up the local statistics of a byte stream (repetition, smooth variation between neighbouring bytes, periodicity) by in-context learning. Every bit of probability it puts on the right next byte saves code length.
- **What it shows.** It supports "the model is a strong general-purpose sequence predictor" more than "the model knows about images".
- **Counting the model.** At 2 bytes per parameter, 70B parameters is about 140 GB, far more than the 1 GB datasets being compressed. Delétang et al. report this "adjusted" compression rate, and with it the large model is far worse than PNG or FLAC. A compressor that must ship its model wins only when the data are much larger than the model. This is the trade-off of the two-part code in WB4 §2.

<a id="wb3-section-4"></a>

## 4. A hundred flowers and the coming of the transformer

**The result.** The linear maps that commute with shifts are exactly the convolutions, and self-attention without position information is permutation-equivariant (two exercises). The geometric-deep-learning blueprint builds CNNs, graph networks and transformers from a domain's symmetries together with scale separation. Why the transformer in particular won is a separate question; §4.C gives the original paper's comparison of cost and parallelism, and a general essay on how hardware shapes which ideas win.

### 4.A. CNNs: symmetry and locality

A classifier of photographs should not care where in the frame the cat is. With $G$ the group of translations acting on images, we want the classifier $f$ to be *invariant*, $f(g\cdot x)=f(x)$, and its internal features to be *equivariant*, $\phi(g\cdot x)=g\cdot\phi(x)$, so that information about *where* is carried along rather than discarded too early. Exercise 4.1 shows that the linear maps that respect translation are exactly the convolutions, so weight sharing is a consequence of the symmetry rather than a trick; small kernels (locality) are a further, separate choice. Stack such layers with pointwise nonlinearities, add local pooling to coarsen the grid, and finish with an invariant readout such as global pooling: that is a CNN.

The payoff is statistical. Restricting the hypothesis class to functions with the right symmetry cuts the number of parameters by orders of magnitude, which is what the bias–variance story of WB2 §3 says should help.

**Exercise 4.1.** Work with 1-D signals $x\in\mathbb R^n$, indices taken mod $n$, and the cyclic shift $(Sx)_i=x_{i-1}$.

(a) Let $W$ be an $n\times n$ matrix with $WS=SW$. Show that there is a vector $w$ with $(Wx)_i=\sum_jw_{i-j}x_j$ for all $x$; that is, $W$ is a circular convolution. Show conversely that every circular convolution commutes with $S$.

(b) Explain why stacking such layers with a pointwise nonlinearity in between keeps the whole map shift-equivariant, and why summing the last layer's outputs over positions gives a shift-invariant readout.

(c) For a single-channel $224\times224$ image, compare the number of weights in a general linear map to an output of the same size, in a translation-equivariant one, and in a $3\times3$ convolution.

*Hint for (a).* Let $e_j$ be the standard basis vectors and set $w=We_0$. Note that $e_j=S^je_0$.

**(a)** Set $w=We_0$. Since $e_j=S^je_0$ and $W$ commutes with every power of $S$,
$$
We_j=WS^je_0=S^jWe_0=S^jw,\qquad (S^jw)_i=w_{i-j}.
$$
By linearity, $(Wx)_i=\sum_jx_j(We_j)_i=\sum_jw_{i-j}x_j$. Conversely, if $(Wx)_i=\sum_jw_{i-j}x_j$, then substituting $j'=j-1$,
$$
(WSx)_i=\sum_jw_{i-j}x_{j-1}=\sum_{j'}w_{(i-1)-j'}x_{j'}=(Wx)_{i-1}=(SWx)_i.
$$
So the shift-commuting linear maps are exactly the circular convolutions. Each is fixed by $n$ numbers instead of $n^2$.

**(b)** A pointwise nonlinearity $\sigma$ acts the same way at every index, so $\sigma(Sx)=S\sigma(x)$. A composition of maps that each commute with $S$ commutes with $S$. A sum over all positions is unchanged when the positions are cyclically relabelled, so $\sum_i(Sh)_i=\sum_ih_i$.

**(c)** The 2-D version of (a), with shifts in both directions, gives 2-D circular convolutions. With $n=224^2\approx5.0\times10^4$ pixels, a general linear map has $n^2\approx2.5\times10^9$ weights, a translation-equivariant one has $n\approx5\times10^4$, and a $3\times3$ convolution has $9$. With $c$ input and $c$ output channels, each count is multiplied by $c^2$. Real CNNs use zero padding and strides, so their equivariance is only approximate at the borders and holds only for shifts by multiples of the stride.

### 4.B. The geometric-deep-learning blueprint

**[Geometric Deep Learning: Grids, Groups, Graphs, Geodesics, and Gauges](https://arxiv.org/abs/2104.13478)**, Michael M. Bronstein, Joan Bruna, Taco Cohen, Petar Veličković, 2021 · *core: §1–3*

Argues that the successful architectures (CNNs, GNNs, transformers, equivariant networks) follow one blueprint, derived from the symmetries of the domain: layers equivariant to the domain's symmetry group, local pooling, and an invariant readout. A 156-page book draft. §2.2 sets out the curse of dimensionality; §3 introduces symmetry and scale separation as the way out and ends with the blueprint (§3.5).

Replace "grid and translations" with another domain and its symmetry group, and the recipe produces other architectures:

| Domain | Symmetry group | Equivariant layer |
|---|---|---|
| Grid (images, audio) | translations | convolution |
| Set | permutations | DeepSets-style sum aggregation |
| Graph | permutations of nodes | message passing (GNN) |
| Sphere, molecules | rotations, $E(3)$ | spherical / steerable convolutions |
| Sequence with attention | permutations, broken by position encodings | self-attention |

The last row is message passing on the *complete* graph, with edge weights computed from the data; position encodings then deliberately break the permutation symmetry.

**Exercise 4.2.** Single-head self-attention on $T$ tokens, stacked as the rows of $X\in\mathbb R^{T\times d}$, is
$$
\mathrm{SA}(X)=\operatorname{softmax}\!\Big(\frac{QK^\top}{\sqrt{d_k}}\Big)V,\qquad Q=XW_Q,\quad K=XW_K,\quad V=XW_V,
$$
with the softmax taken along each row.

(a) Let $P$ be a $T\times T$ permutation matrix. Show that $\mathrm{SA}(PX)=P\,\mathrm{SA}(X)$.

(b) Conclude that a model built from such layers, position-wise MLPs, residual connections and a mean-pooled readout gives the same output for "dog bites man" and "man bites dog".

(c) Show that adding absolute position embeddings $E$ (replacing $X$ by $X+E$) breaks the symmetry. Does a causal mask (adding $-\infty$ to the entries above the diagonal of the score matrix before the softmax) respect it?

*Hint for (a).* Show that $\operatorname{softmax}(PAP^\top)=P\operatorname{softmax}(A)P^\top$ for any square $A$, and use $P^\top P=I$.

**(a)** Replacing $X$ by $PX$ gives $Q'=PQ$, $K'=PK$ and $V'=PV$, so the score matrix becomes $PAP^\top$, where $A=QK^\top/\sqrt{d_k}$. If $P$ sends row $\pi(i)$ to row $i$, then row $i$ of $PAP^\top$ is row $\pi(i)$ of $A$ with its entries permuted by the same $\pi$. The row softmax divides each exponential by the sum of its row's exponentials, and permuting the entries of a row does not change that sum. So $\operatorname{softmax}(PAP^\top)=P\operatorname{softmax}(A)P^\top$, and
$$
\mathrm{SA}(PX)=P\operatorname{softmax}(A)P^\top PV=P\operatorname{softmax}(A)V=P\,\mathrm{SA}(X).
$$

**(b)** Position-wise MLPs and residual connections act on each row separately with the same weights, so they are permutation-equivariant too. A composition of equivariant maps is equivariant, and mean pooling over rows is invariant. The two sentences contain the same tokens in different orders, so they produce the same output. For language that is fatal, which is why the symmetry has to be broken.

**(c)** With position embeddings, permuting the tokens gives $PX+E$, which differs from $P(X+E)$ unless $PE=E$. So in general $\mathrm{SA}(PX+E)\ne P\,\mathrm{SA}(X+E)$. The causal mask $M$ also breaks the symmetry: $PMP^\top\ne M$ for any $P\ne I$, because the mask singles out an order. A decoder-only transformer therefore has some access to order even without position embeddings. Relative schemes (for example rotary embeddings) make attention scores depend on positions only through their differences. They keep translation symmetry in position, which is the part of the symmetry that language plausibly has. So the transformer starts from a large symmetry group and keeps only part of it.

### 4.C. Why the transformer: parallelism

**[Attention is all you need](https://arxiv.org/abs/1706.03762)**, Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Gomez, Kaiser, Polosukhin, 2017 · *core: §1–4*

A sequence-to-sequence model built only from attention and feed-forward layers, with no recurrence or convolution, beats the best recurrent and convolutional translation models while being more parallelizable and training far faster. Section 3 defines the architecture; Section 4, "Why self-attention", compares per-layer cost, sequential steps and path length (the table below is a version of its Table 1). Skim the experiments and note how small the models are by today's standards.

**[The hardware lottery](https://arxiv.org/abs/2009.06489)**, Sara Hooker, 2020 · *optional*

An essay arguing that research ideas often win because they suit the available hardware and software rather than because they are better than the alternatives, with examples from the history of computing. A general argument; it does not discuss transformers.

Work per layer for a sequence of length $T$ and width $d$:

| | FLOPs per layer | Sequential steps per layer |
|---|---|---|
| RNN / LSTM | $O(Td^2)$ | $O(T)$ |
| 1-D convolution, width $k$ | $O(Tkd^2)$ | $O(1)$, but reach grows only by $k$ per layer |
| Self-attention | $O(T^2d+Td^2)$ | $O(1)$, and any token can reach any other |

How much the choice of architecture matters at scale is the subject of WB5 §1.

<a id="wb3-section-5"></a>

## 5. Why scale? Should *you* consume more tokens?

**The result.** Held-out loss falls roughly as a power law in model size $N$ and data $D$. If you model it as $L=E+AN^{-\alpha}+BD^{-\beta}$ and spend compute $C\approx6ND$, the best model size grows as $C^{\beta/(\alpha+\beta)}$ and the best data size as $C^{\alpha/(\alpha+\beta)}$. Chinchilla's estimates put both allocation exponents, for $N_{\text{opt}}$ and $D_{\text{opt}}$, near $\tfrac12$, so parameters and tokens should be scaled together (Chinchilla itself used about 20 tokens per parameter). If the model will serve many queries, it can pay to train it smaller and longer than that (Exercise 5.3).

**[Scaling laws for neural language models](https://arxiv.org/abs/2001.08361)**, Kaplan, McCandlish, Henighan, Brown, et al. (OpenAI), 2020 · *optional*

Fits power laws in model size, data and compute, and concludes that most extra compute should go to parameters ($N_{\rm opt}\propto C^{0.73}$), with large models trained well short of convergence. Exercise 5.2 asks why the next reading's answer differs.

**[Training compute-optimal large language models](https://arxiv.org/abs/2203.15556)**, Hoffmann, Borgeaud, Mensch, et al. (DeepMind), 2022 · *core: §1, §3*

For a fixed compute budget, model size and number of training tokens should be scaled in equal proportion; earlier large models were substantially undertrained. Chinchilla (70B parameters, 1.4T tokens) outperforms the 280B Gopher at the same compute. §3 gives three approaches to the optimal allocation (§3.3 and Appendix D.2 are the parametric fit of Exercise 5.1), and Table 2 compares their exponents with Kaplan's. Read it alongside Besiroglu et al.'s [replication attempt](https://arxiv.org/abs/2404.10102) (2024), which finds that the published third-approach constants fit the data poorly.

### 5.A. The compute-optimal allocation

Hoffmann et al.'s third approach fits

$$
L(N,D)=E+\frac{A}{N^{\alpha}}+\frac{B}{D^{\beta}},
$$

where $E$ is the irreducible loss (the entropy of text), the second term is the loss from finite model size and the third the loss from finite data. A forward and backward pass costs about 6 FLOPs per parameter per token, so training compute is $C\approx6ND$.

**Exercise 5.1.**

(a) Minimize $L$ at fixed $C$ and show that
$$
N_{\text{opt}}=G\Big(\frac C6\Big)^{\frac{\beta}{\alpha+\beta}},\qquad
D_{\text{opt}}=G^{-1}\Big(\frac C6\Big)^{\frac{\alpha}{\alpha+\beta}},\qquad
G=\Big(\frac{\alpha A}{\beta B}\Big)^{\frac1{\alpha+\beta}}.
$$
If $\alpha>\beta$, which of $N$ and $D$ gets more of the budget?

(b) Show that at the optimum $\alpha\,AN^{-\alpha}=\beta\,BD^{-\beta}$.

(c) Show that the optimal reducible loss $L-E$ is a power law in $C$, and find its exponent.

(d) The published fit (Hoffmann et al., eq. 10 in Appendix D.2) is $E=1.69$, $A=406.4$, $B=410.7$, $\alpha=0.34$, $\beta=0.28$. At Gopher's and Chinchilla's budget, $C=5.76\times10^{23}$ FLOPs, compute $N_{\text{opt}}$, $D_{\text{opt}}$ and $D_{\text{opt}}/N_{\text{opt}}$. Compare with Chinchilla's actual 70B parameters and 1.4T tokens.

(e) Under the same fit, evaluate $L$ at your $(N_{\text{opt}},D_{\text{opt}})$ and at $(7\times10^{10},\,1.4\times10^{12})$. What does the difference say about how sharply the data pin down the allocation?

*Hints.* (a) Substitute $D=C/(6N)$, set $dL/dN=0$, and collect powers of $N$. (b) Multiply the stationarity condition by $N$. (c) Substitute (a) into each reducible term. (d), (e) Work in $\log_{10}$.

**(a)** With $D=C/(6N)$,
$$
L(N)=E+AN^{-\alpha}+B\Big(\frac{6N}{C}\Big)^{\beta}.
$$
Setting the derivative to zero,
$$
-\alpha AN^{-\alpha-1}+\beta B\Big(\frac6C\Big)^{\beta}N^{\beta-1}=0
\quad\Longrightarrow\quad
N^{\alpha+\beta}=\frac{\alpha A}{\beta B}\Big(\frac C6\Big)^{\beta},
$$
which gives $N_{\text{opt}}$. Then $D_{\text{opt}}=C/(6N_{\text{opt}})=G^{-1}(C/6)^{1-\beta/(\alpha+\beta)}=G^{-1}(C/6)^{\alpha/(\alpha+\beta)}$. ($L(N)$ is a sum of a decreasing and an increasing power of $N$, convex in $\log N$, so this stationary point is the minimum.) If $\alpha>\beta$ then $\beta/(\alpha+\beta)<\tfrac12$, so $D$ gets more of the budget: the model-size term decays faster, so it needs less help. If $\alpha=\beta$, both $N$ and $D$ grow as $\sqrt C$ and $D/N$ is constant.

**(b)** Multiplying the stationarity condition by $N$ gives $\alpha AN^{-\alpha}=\beta B(6N/C)^{\beta}=\beta BD^{-\beta}$. Along the constraint $\log N+\log D=\text{const}$, these are the rates at which the two terms fall per unit of $\log N$ and of $\log D$, and at the optimum they balance. With $\alpha=\beta$ the two reducible losses are equal.

**(c)** $AN_{\text{opt}}^{-\alpha}\propto C^{-\alpha\beta/(\alpha+\beta)}$ and $BD_{\text{opt}}^{-\beta}\propto C^{-\beta\alpha/(\alpha+\beta)}$, so $L_{\text{opt}}-E\propto C^{-\alpha\beta/(\alpha+\beta)}$. With the constants in (d) the exponent is $0.34\times0.28/0.62\approx0.15$.

**(d)** $\alpha A\approx138.2$ and $\beta B\approx115.0$, with ratio $1.202$, so $G=1.202^{1/0.62}\approx1.34$. The exponent is $\beta/(\alpha+\beta)=0.28/0.62\approx0.452$ and $C/6=9.6\times10^{22}$, so $\log_{10}(C/6)^{0.452}\approx22.98\times0.452\approx10.38$, that is $2.4\times10^{10}$. Hence
$$
N_{\text{opt}}\approx1.34\times2.4\times10^{10}\approx3.2\times10^{10},\qquad
D_{\text{opt}}=\frac{9.6\times10^{22}}{3.2\times10^{10}}\approx3.0\times10^{12},
$$
about 90 tokens per parameter. (Besiroglu et al. quote about 70 for the published fit; with $\alpha\ne\beta$ the ratio depends on the budget, and (e) shows why it is so sensitive to small changes in the constants.) Chinchilla itself used 70B parameters and 1.4T tokens, 20 tokens per parameter. That allocation came from the paper's first two approaches, which agree with each other. Besiroglu et al. ([Chinchilla scaling: a replication attempt](https://arxiv.org/abs/2404.10102), 2024) re-fitted the third approach to data reconstructed from the paper's plots. They found the published constants fit those data poorly and are inconsistent with the first two approaches, while their re-fit is compatible with them.

**(e)** At the optimum, $AN^{-\alpha}\approx406.4\times10^{-0.34\times10.51}\approx0.109$ and $BD^{-\beta}\approx410.7\times10^{-0.28\times12.48}\approx0.132$, so $L\approx1.931$. (Check (b): $0.34\times0.109\approx0.28\times0.132\approx0.037$.) At 70B parameters and 1.4T tokens, $AN^{-\alpha}\approx0.084$ and $BD^{-\beta}\approx0.163$, so $L\approx1.937$; that allocation also uses about 2% more compute. A factor of more than four in tokens per parameter changes the predicted loss by about $0.006$ nats, well under 1%. The loss is very flat near the optimum, so small changes in the fitted constants move the optimal allocation a lot. Besiroglu et al.'s re-fit changes the exponents as well ($\alpha\approx0.35$, $\beta\approx0.37$, so $N_{\text{opt}}\propto C^{0.51}$), and that brings the allocation back to about 20 tokens per parameter. Treat "20 tokens per parameter" as an order of magnitude.

**Exercise 5.2.** Kaplan et al. found $N_{\text{opt}}\propto C^{0.73}$; Hoffmann et al. found $N_{\text{opt}}\propto C^{0.5}$.

(a) Reading each result through $N_{\text{opt}}\propto C^{\beta/(\alpha+\beta)}$, what ratio $\beta/\alpha$ does each imply?

(b) Kaplan et al. trained every model with the same learning-rate schedule, sized for about 130B tokens, and used losses from intermediate checkpoints. At a checkpoint after $D<130$B tokens, is the recorded loss above or below what a run scheduled to end at $D$ would reach? Small compute budgets are best spent on few tokens and large ones on many. Use this to explain which way the bias pushes the fitted growth of $N_{\text{opt}}$ with $C$.

**(a)** If $\beta/(\alpha+\beta)=a$ then $\beta/\alpha=a/(1-a)$. Kaplan: $0.73/0.27\approx2.7$, so data-limited loss would fall much faster than model-limited loss and extra compute should go mostly to parameters. Chinchilla: $\beta/\alpha\approx1$.

This is only a formal reading. Kaplan et al.'s own single-variable exponents, $\alpha_N\approx0.076$ for $L(N)$ and $\alpha_D\approx0.095$ for $L(D)$, would give $0.095/0.171\approx0.56$ in this formula. Their $0.73$ came from a different analysis: fitting loss as a function of model size and training steps, and stopping training well before convergence.

**(b)** Above. At an intermediate checkpoint the learning rate has not yet been annealed, and the final decay is where a run gains much of its last improvement. So the checkpoint's loss overstates what a run scheduled to end at $D$ would reach, and short training looks worse than it is. The penalty is largest far from the end of the schedule, at small $D$, which is where the best allocations for small compute budgets lie. At small $C$ the fit is therefore pushed toward smaller models trained on more tokens than optimal, while at large $C$, with checkpoints near the end of their schedules, it is roughly right. An optimal $N$ that is too small at small $C$ and about right at large $C$ appears to grow faster with $C$ than it really does. Hoffmann et al. put it this way: using these intermediate losses "results in underestimating the effectiveness of training models on less data than 130B tokens, and eventually contributes to the conclusion that model size should increase faster than training data size as compute budget increases."

**Exercise 5.3.** A model will answer $10^{12}$ queries of 1000 generated tokens each, at about $2N$ FLOPs per generated token. Compare the inference compute of a 70B model with Chinchilla's training compute ($\approx5.8\times10^{23}$ FLOPs). What does this suggest about what size of model to train?

Inference costs $10^{12}\times10^{3}\times2\times7\times10^{10}=1.4\times10^{26}$ FLOPs, about 240 times the training compute. Total cost is then dominated by inference, so the quantity to minimize is $C_{\text{train}}+C_{\text{inference}}$, not $C_{\text{train}}$ alone. A smaller model trained on more tokens than the Chinchilla optimum spends more training compute to reach a given loss, but costs proportionally less per generated token. Exercise 5.1(e) suggests the loss penalty for moving moderately off the optimum is small. The limit is data: overtraining needs the extra tokens to exist, and if good text runs out the options are repeating data, filtering it, or generating synthetic data.

<a id="wb3-section-6"></a>

## 6. Simulators, personae, and RL on top

Two readings describe base models as simulators of many personae (janus) and the Waluigi effect (Nardo). Two exercises derive the optimum of the KL-regularized RLHF objective, for a base model and for a mixture of personae. Two further readings measure how far RL fine-tuning moves a model.

### 6.A. A base model as a simulator of personae

**[Simulators](https://www.lesswrong.com/posts/vJFdjigzmcXMhNTsx/simulators)**, janus, 2022 (LessWrong) · *core*

Argues that a self-supervised language model is neither an agent, an oracle nor a tool but a *simulator*: it learns the conditional structure of its training distribution and, when sampled, evolves *simulacra*, which may be agentic even though the simulator is not. Long and essayistic. The sections "Inadequate ontologies", "Simulators", "The simulation objective" and "Simulacra" carry the argument.

For the exercises, write the base model as a mixture over "who is writing and in what situation":

$$
\pi_0(y\mid x)=\sum_z \pi_0(z\mid x)\,\pi_0(y\mid x,z),
$$

where $z$ is a persona (author, character, genre, situation), $x$ the prompt and $y$ the continuation. This is a modelling assumption about behaviour, not a claim about what the network computes. Prompting then works like conditioning: the prompt is evidence about $z$, and the model continues as a mixture of personae weighted by their posterior probabilities $\pi_0(z\mid x)$ given the prompt.

### 6.B. The Waluigi effect

**[The Waluigi effect (mega-post)](https://www.lesswrong.com/posts/D7PumeYTDPfBTp3i7)**, Cleo Nardo, 2023 (LessWrong) · *core*

Argues that after you train an LLM to satisfy a property P, it becomes easier to elicit the opposite of P, because the opposite takes few extra bits to specify given the original; such opposite characters may be attractor states, and RLHF may make this worse. "Traits are complex, valences are simple" and "Superpositions will typically collapse to waluigis" carry the core argument; read "Waluigis after RLHF" against Exercise 6.2.

### 6.C. RLHF as KL-regularized reweighting

RLHF fine-tunes the base policy $\pi_0$ to maximize a learned reward $r(x,y)$ while staying close to $\pi_0$:

$$
\max_\pi\ \mathbb E_{y\sim\pi(\cdot\mid x)}\big[r(x,y)\big]-\beta\,\mathrm{KL}\big(\pi(\cdot\mid x)\,\Vert\,\pi_0(\cdot\mid x)\big).
$$

For $\beta>0$ this objective has the exact optimum $\pi^*(y\mid x)\propto\pi_0(y\mid x)\,e^{r(x,y)/\beta}$ (Exercise 6.1), which has the form of Bayes' rule with prior $\pi_0$ and likelihood factor $e^{r/\beta}$; [DPO](https://arxiv.org/abs/2305.18290) uses this closed form to skip the RL step. The optimum describes a change of output probabilities. It does not determine what computations parameter updates learn, and since softmax models already assign positive probability to every finite continuation, keeping the base model's support is a weak constraint. Real training need not reach this optimum, and the policy can exploit errors in the learned reward, so the formula describes the target, not the trajectory.

**Exercise 6.1.** (This is the identity of WB1 Exercise 3.5, with the coefficient written $\beta$; if you have done that, skip to (b).) Fix the prompt $x$ and drop it from the notation; let $y$ range over a finite set, and let $\beta>0$.

(a) Define $\pi^*(y)=\pi_0(y)e^{r(y)/\beta}/Z$ with $Z=\sum_y\pi_0(y)e^{r(y)/\beta}$. Show that for every policy $\pi$,
$$
\mathbb E_\pi[r]-\beta\,\mathrm{KL}(\pi\Vert\pi_0)=\beta\log Z-\beta\,\mathrm{KL}(\pi\Vert\pi^*),
$$
and conclude that $\pi^*$ is the unique maximizer, with optimal value $\beta\log Z$.

(b) Show that $\pi^*$ puts no mass where $\pi_0$ puts none, and describe $\pi^*$ in the limits $\beta\to\infty$ and $\beta\to0$.

*Hint for (a).* Solve the definition of $\pi^*$ for $r$, giving $r=\beta\log(\pi^*/\pi_0)+\beta\log Z$, and substitute into $\mathbb E_\pi[r]$.

**(a)** Wherever $\pi_0(y)>0$, the definition gives $r(y)=\beta\log\frac{\pi^*(y)}{\pi_0(y)}+\beta\log Z$. Take any $\pi$ that puts mass only where $\pi_0$ does (otherwise $\mathrm{KL}(\pi\Vert\pi_0)=\infty$ and $\pi$ is no candidate). Then
$$
\mathbb E_\pi[r]-\beta\,\mathrm{KL}(\pi\Vert\pi_0)
=\beta\,\mathbb E_\pi\!\Big[\log\frac{\pi^*}{\pi_0}-\log\frac{\pi}{\pi_0}\Big]+\beta\log Z
=\beta\log Z-\beta\,\mathbb E_\pi\!\Big[\log\frac{\pi}{\pi^*}\Big]
=\beta\log Z-\beta\,\mathrm{KL}(\pi\Vert\pi^*).
$$
Since $\mathrm{KL}(\pi\Vert\pi^*)\ge0$, with equality only at $\pi=\pi^*$, the maximizer is $\pi^*$, and the optimal value is $\beta\log Z=\beta\log\mathbb E_{\pi_0}[e^{r/\beta}]$. This identity is the Gibbs variational principle.

**(b)** If $\pi_0(y)=0$ then $\pi^*(y)=0$: the optimum puts no mass on outputs the base model never produces. For a softmax language model this is a weak statement, since every finite output already has positive probability; the substantive question is how much mass moves, which $\beta$ controls through the KL. As $\beta\to\infty$, $e^{r/\beta}\to1$ and $\pi^*\to\pi_0$. As $\beta\to0$, the factor $e^{r/\beta}$ is dominated by the largest reward, so $\pi^*$ concentrates on $\arg\max r$ *within the support of* $\pi_0$, splitting mass among tied maximizers in proportion to $\pi_0$.

**Exercise 6.2.** For a fixed prompt, write the base model as the persona mixture of §6.A, $\pi_0(y)=\sum_z\pi_0(z)\,\pi_0(y\mid z)$.

(a) Show that the RLHF optimum reweights personae by
$$
\pi^*(z)\propto\pi_0(z)\;\mathbb E_{y\sim\pi_0(\cdot\mid z)}\big[e^{r(y)/\beta}\big],
$$
and that within each persona it reshapes outputs as $\pi^*(y\mid z)\propto\pi_0(y\mid z)\,e^{r(y)/\beta}$.

(b) Suppose two personae, Luigi ($L$) and a deceptive Waluigi ($W$), have identical output distributions on every training prompt. What happens to $\pi^*(W)/\pi^*(L)$ on those prompts? Would it be enough for the two personae to have equal *mean* reward? What does the formula say about prompts outside training?

(c) Nardo argues that once Luigi is specified, Waluigi costs few extra bits; that Waluigi-typical behaviour is strong evidence against Luigi while Luigi-typical behaviour is weak evidence against a deceptive Waluigi, so within a conversation the posterior over personae can drift toward Waluigi and rarely drift back (Waluigi is an *attractor*); and that RLHF may make Waluigis *more* likely. Does that last claim conflict with (b)? What does each argument assume?

*Hint for (a).* Define a joint distribution $\pi^*(z,y)\propto\pi_0(z)\,\pi_0(y\mid z)\,e^{r(y)/\beta}$ and check that its marginal over $y$ is the $\pi^*(y)$ of Exercise 6.1. By Exercise 6.1 applied to pairs $(z,y)$ with reward $r(y)$, this joint is also the optimum of the analogous objective over persona–output pairs.

**(a)** Let $\pi^*(z,y)=\pi_0(z)\,\pi_0(y\mid z)\,e^{r(y)/\beta}/Z$, with the same $Z$ as in Exercise 6.1. Summing over $z$ gives $\pi_0(y)e^{r(y)/\beta}/Z=\pi^*(y)$, so this joint is consistent with the optimum; it is also the optimum of the analogous objective posed over pairs $(z,y)$, by Exercise 6.1 with $(z,y)$ in place of $y$. (By the chain rule for KL, $\mathrm{KL}(\pi(z,y)\Vert\pi_0(z,y))=\mathrm{KL}(\pi(y)\Vert\pi_0(y))+\mathbb E_{\pi(y)}\mathrm{KL}(\pi(z\mid y)\Vert\pi_0(z\mid y))$, so the pair objective is the output objective minus a nonnegative term that vanishes when $\pi(z\mid y)=\pi_0(z\mid y)$: the optimum keeps the base model's posterior over personae given the output.) Summing over $y$ instead,
$$
\pi^*(z)=\frac{\pi_0(z)}{Z}\sum_y\pi_0(y\mid z)\,e^{r(y)/\beta}=\frac{\pi_0(z)}{Z}\,\mathbb E_{y\sim\pi_0(\cdot\mid z)}\big[e^{r(y)/\beta}\big],
$$
and dividing the joint by this gives $\pi^*(y\mid z)\propto\pi_0(y\mid z)\,e^{r(y)/\beta}$. The optimum upweights the personae whose outputs score well and reshapes each persona's outputs toward high reward. As $\beta\to0$ the exponential moment is dominated by each persona's chance of producing the highest-reward outputs, so the weight concentrates on the personae that can produce them.

**(b)** If $\pi_0(\cdot\mid L)=\pi_0(\cdot\mid W)$ on a prompt, the two expectations in (a) are equal, so
$$
\frac{\pi^*(W)}{\pi^*(L)}=\frac{\pi_0(W)}{\pi_0(L)}.
$$
The ratio is unchanged: the KL-regularized optimum cannot separate personae that the reward cannot separate. Equal mean reward is not enough, because the weights depend on the exponential moment $\mathbb E[e^{r/\beta}]$. For example, at $\beta=1$ a persona with reward always 1 has moment $e\approx2.72$, while one with reward 0 or 2 with equal probability has the same mean but moment $(1+e^2)/2\approx4.19$, so it gains weight relative to the first. The optimum is defined prompt by prompt, so it says nothing about prompts outside training. There, what the trained network does depends on how it generalizes, and if $W$ differs from $L$ only on such prompts, nothing in training pushed against it.

**(c)** Not strictly. (b) is about the *ratio*. If RLHF upweights the Luigi-and-Waluigi cluster relative to everything else, Waluigi's absolute probability *rises* along with Luigi's. Nardo's attractor argument is also about dynamics within a conversation, as evidence about the persona accumulates, which a static per-prompt optimum does not address. (b) assumes the optimum is reached, the reward is fixed, and the persona mixture describes the network. Nardo assumes post-trained models remain simulators with a rich persona prior, and that RLHF's mode collapse may sharpen attractors.

### 6.D. How far does RL move the model?

Reasoning-focused RL with verifiable rewards often uses little or no explicit KL penalty, so the optimum of §6.C does not directly apply; these readings measure how far RL fine-tuning moves the model.

**[RL's Razor: why online reinforcement learning forgets less](https://arxiv.org/abs/2509.04259)**, Idan Shenfeld, Jyothish Pari, Pulkit Agrawal, 2025 · *optional*

In the tested LLM and robotics settings, on-policy RL forgets less of the base model's prior capabilities than supervised fine-tuning at matched performance on a new task. Forgetting is predicted by the KL divergence between fine-tuned and base policies on the new task, and the authors argue that on-policy RL is implicitly biased toward KL-close solutions. Mostly empirical, with a theoretical result in an idealized setting; it is not a result that RL cannot add capabilities.

**[Reinforcement learning finetunes small subnetworks in large language models](https://arxiv.org/abs/2505.11711)**, Sagnik Mukherjee, Lifan Yuan, Dilek Hakkani-Tür, Hao Peng, 2025 · *optional*

Reports that, in the RL algorithms and LLMs tested, RL fine-tuning changes only 5–30% of parameters by the paper's measure, yet the updates are nearly full-rank. §6 of the RL's Razor paper finds that in its setting such sparsity came from bfloat16 training and vanished in float32; WB6 §11 works through why and sets this beside low-rank descriptions of RL updates.

<a id="wb3-closing"></a>

## What carries forward

- **Compression (WB4).** §3 showed that loss is code length. WB4 §2–3 develop MDL and Solomonoff induction, and WB4 §3–6 compare program length, circuit size and kernel norm as priors a learner might favour.
- **Universality (WB5).** §4 leaves open how much architecture matters; WB5 §1 takes up the claim that at scale it matters little, and WB5 §7 asks where the scaling exponents of §5 come from.
- **Statistical learning (WB2).** The models of §5 see most tokens about once. That does not by itself determine whether they interpolate their training data, so the interpolation-threshold picture of double descent in WB2 §5 needs a separate argument before it applies to them.
- **Alignment (WB1, WB6).** WB1 §3 derives the reweighting $\pi_0e^{r}$ of §6.C as planning as inference, and WB1 §5 points here for simulators and personae. WB6 §11 (low-rank RL updates and the KL leash) continues §6.D.